In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import glob
import pandas as pd
import shutil

# 1. Define Paths
source_folder = '/content/drive/MyDrive/CPTAC-LSCC/PKG - CPTAC-LSCC_v10/LSCC'
output_base_folder = '/content/drive/MyDrive/CPTAC-LSCC'
tumor_folder = os.path.join(output_base_folder, 'TUMOR')
normal_folder = os.path.join(output_base_folder, 'NORMAL')

csv_path = os.path.join(output_base_folder, 'CPTAC_Data.csv')

print("Loading TCIA Master Table...")
try:
    # 2. Read the CSV
    df = pd.read_csv(csv_path)

    # Clean the column to avoid spacing errors, then filter
    df['Tumor'] = df['Tumor'].astype(str).str.strip()
    df_lscc = df[df['Tumor'] == 'LSCC']

    print(f"✅ Dropped other cancers. Using {len(df_lscc)} LSCC-specific records (out of {len(df)} total).")

    # 3. Create the dictionary using ONLY the filtered LSCC data
    master_dict = dict(zip(
        df_lscc['Slide_ID'].astype(str).str.strip(),
        df_lscc['Specimen_Type'].astype(str).str.strip()
    ))

    print(f"✅ Loaded {len(master_dict)} slide labels from the table.\n")

    # 4. Get all remaining .svs files that failed the API check
    unsorted_files = glob.glob(os.path.join(source_folder, '*.svs'))

    if not unsorted_files:
        print("🎉 No more files left to sort in the LSCC folder!")
    else:
        print(f"🧹 Found {len(unsorted_files)} files left behind. Cleaning up...\n")

        moved_tumor = 0
        moved_normal = 0
        still_unknown = 0

        for file_path in unsorted_files:
            filename = os.path.basename(file_path)
            slide_id = filename.replace('.svs', '')

            # Look up the slide in our new dictionary
            label = str(master_dict.get(slide_id, "Unknown")).lower()

            if "tumor" in label:
                shutil.move(file_path, os.path.join(tumor_folder, filename))
                print(f"✅ Moved {filename} ➡️ TUMOR")
                moved_tumor += 1
            elif "normal" in label:
                shutil.move(file_path, os.path.join(normal_folder, filename))
                print(f"✅ Moved {filename} ➡️ NORMAL")
                moved_normal += 1
            else:
                print(f"⚠️ Unknown in CSV: {filename}")
                still_unknown += 1

        print("\nCLEANUP COMPLETE!")
        print(f"Tumor: {moved_tumor}")
        print(f"Normal: {moved_normal}")
        if still_unknown > 0:
            print(f"Still stuck: {still_unknown}")

except FileNotFoundError:
    print(f"❌ Error: Could not find the CSV file at {csv_path}")
    print("Check the file name and make sure it is uploaded to your Drive!")
except KeyError as e:
    print(f"❌ Error: Could not find the expected column {e} in your CSV.")
    print("Open the CSV and make sure the headers are exactly 'Slide_ID' and 'Specimen Type'.")